In [5]:
# Load necessary libraries and clean the dataset

import pandas as pd
from scipy.stats import ttest_ind
import statsmodels.api as sm

# Load the dataset
df = pd.read_csv("dodgers-2022.csv")

# Strip whitespace and standardize column names
df.columns = df.columns.str.strip().str.lower()

# Convert YES and NO responses to binary
for col in['fireworks', 'bobblehead', 'shirt', 'cap']:
    df[col] = df[col].str.strip().map({'YES': 1, 'NO': 0})

# Create a datetime column
month_map = {'APR': 4, 'MAY': 5, 'JUN': 6, 'JUL': 7, 'AUG': 8, 'SEP': 9, 'OCT': 10}
df['month_num'] = df['month'].map(month_map)
df['date'] = pd.to_datetime(dict(year=2022, month=df['month_num'], day=df['day']))

In [9]:
# Function to evaluate the impact of a promotion on attendance
def promotion_effect(col):
    promotion = df[df[col] == 1]['attend']
    no_promo = df[df[col] == 0]['attend']
    t_stat, p_val = ttest_ind(promotion, no_promo, equal_var=False)
    print(f"{col.capitalize()} Promotion:")
    print(f" The Mean with Promotion(s): {promotion.mean():.2f}")
    print(f" The Mean without Promotion(s): {no_promo.mean():.2f}")
    print(f" t-statistic: {t_stat:.2f}, p-value: {p_val:.2f}\n")

# Loop through each promotion type and run the analysis
for col in ['fireworks', 'bobblehead', 'shirt', 'cap']:
    promotion_effect(col)

Fireworks Promotion:
 The Mean with Promotion(s): 41077.86
 The Mean without Promotion(s): 41032.18
 t-statistic: 0.02, p-value: 0.98

Bobblehead Promotion:
 The Mean with Promotion(s): 53144.64
 The Mean without Promotion(s): 39137.93
 t-statistic: 11.46, p-value: 0.00

Shirt Promotion:
 The Mean with Promotion(s): 46643.67
 The Mean without Promotion(s): 40824.55
 t-statistic: 1.82, p-value: 0.19

Cap Promotion:
 The Mean with Promotion(s): 38189.50
 The Mean without Promotion(s): 41112.24
 t-statistic: -0.66, p-value: 0.62



Assignment Instructions: Tell a story with your analysis and clearly explain the steps you take to arrive at your conclusion...clearly state any assumptions you make.

First, let's interpret the results here. The Fireworks Promotion has a mean attendance "increase" of approximately 45.68 attendees and a not statistically significant p-value of 0.98. This means that the Fireworks Promotion days do not reveal much impact on the attendance in this dataset. When speaking to management, I would recommend that they reevaluate the cost-effectiveness of fireworks usage as an attendance draw.

The Bobblehead Promotion has a mean attendance increase of 14,006.71 attendees and a highly statistically significant p-value of 0.00. This means that the Bobblehead Promotion is a huge driver of attendance. My recommendation to management would be to expand the Bobblehead Promotion, especially on days of the week that see attendance drops, to entice attendees on days that aren't quite crowd drawers.

The Shirt Promotion has a mean attendance increase of 5,819.12 attendees and a 0.19 p-value, not statistically significant. This means that the Shirt Promotion, compared to the Bobblehead Promotion so far, reveals a moderate increase in attendees without the strong statistical effect. I would recommend to management to pair the Shirt Promotion with the lesser-performing promotion of fireworks to boost its impact.

The Cap Promotion has a mean attendance decrease of 2,922.74 attendees and a p-value of 0.62, not statistically significant. This reveals that the Cap Promotion not only does not attract additional attendees, but it might even result in a lower turnout. My recommendation to management would be to cease this promotion or try to pair it with the Fireworks promotion in an attempt to boost its impact.


In [7]:
# Build an Ordinary Least Squares (OLS) Regression Model to quantify the impact of promotion type on attendance
X = df[['fireworks', 'bobblehead', 'shirt', 'cap']]
X = sm.add_constant(X)
y = df['attend']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 attend   R-squared:                       0.387
Model:                            OLS   Adj. R-squared:                  0.355
Method:                 Least Squares   F-statistic:                     12.01
Date:                Sat, 27 Sep 2025   Prob (F-statistic):           1.29e-07
Time:                        10:08:22   Log-Likelihood:                -825.51
No. Observations:                  81   AIC:                             1661.
Df Residuals:                      76   BIC:                             1673.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        3.82e+04    933.092     40.940      0.0

Assignment Instructions: Tell a story with your analysis and clearly explain the steps you take to arrive at your conclusion...clearly state any assumptions you make.

Here are the regression results that reinforce the statistical findings from earlier and give a quantifiable way to communicate the impact of each promotion. In overview, R-squared is 0.387, meaning the model explains nearly 39% of the variation in attendance with just the four promotion types I selected. The F-statistic p-value of 1.29e-07 reveals that the model is statistically significant overall; this is great news.

Some of the individual effects of the promotions:

1. fireworks, coefficient of 2,877, p-value 0.157, a small and not statistically significant change to attendance
2. bobblehead, coefficient of 14,940, p-value 0.000, showing a strong and significant boost in attendance
3. shirt, coefficient of 8,443, p-value 0.036, a moderate and statistically significant boost to attendance
4. cap, coefficient of -12, p-value 0.998, no effect on attendance

Bobbleheads as a promotion are the clear winner with an increase of 14,940 attendees per game and strong statistical significance. Shirts as a promotion are impactful and may be worth expanding upon when paired with weather that suits the clothing. Fireworks see a modest increase, but the effect of this isn't statistically reliable, and may work better with other promotions. Caps as a promotion do not affect attendance and should be reconsidered as a promotion or combined with others to boost their impact.

Conclusion:

The goal of this analysis was to find what drives game attendance and translate those insights into actionable insights for management. I was immediately drawn to cap, shirt, fireworks, and bobblehead as columns. This is based on the assumption that people respond to value, and nothing adds value to a purchase more than additional free "stuff" or promotions. So I knew what I had to do: look into how cap, shirt, fireworks, and bobblehead impact attendance.

I started with basic statistical comparisons, the mean with and without promotion, t-statistic, and p-value for each promotion to assess significance. To deepen the analysis, I built an OLS Regression model to quantify the individual impact of each promotion while controlling for the others.

Ultimately, my plan paid off, revealing an R-squared of 0.387, or 39%, which is significant and an excellent place for me to explain to management where they can start making changes. The biggest takeaways were that the Bobblehead Promotion was most effective, seeing over a 14,000 attendee increase, while the Shirt Promotion saw a meaningful boost. In contrast, the Cap Promotion needs rethinking to bring in the attendees, and the Fireworks Promotion, while showing an attendance increase, didn't significantly affect attendance.

These findings give management actionable insights and a plan forward; more bobbleheads, shirts, rethink the fireworks and caps.